# Benchmarking and Evaluating Qwen3-0.6B: BF16 vs GPTQ-W8A16

This notebook measures two versions of the same model, served with **vLLM** on a local GPU:

| Model | Folder | Weights |
|:--|:--|:--|
| **Qwen3-0.6B** | `Models/Qwen3-0.6B` | Original BF16 (16-bit) |
| **Qwen3-0.6B GPTQ-W8A16** | `Models/Qwen3-0.6B-GPTQ-W8A16` | 8-bit integer weights, 16-bit activations |

A quantized model is only worth deploying if it is **faster or cheaper** *and* still gives **good answers**, so each model is measured in both ways:

| Dimension | Question | Tool |
|:--|:--|:--|
| **Performance** | How fast does the server answer, and how much load can it handle? | [GuideLLM](https://github.com/vllm-project/guidellm) |
| **Quality** | How accurate is the model on standard benchmark tasks? | [lm_eval](https://github.com/EleutherAI/lm-evaluation-harness) |

Both tools talk to the vLLM server over HTTP, the same way a real application would. The notebook serves one model, runs both tools against it, stops it, then repeats with the other model. At the end, the results are compared side by side.

## Before you start

- Run this notebook with the **Python (vllm-gpu)** kernel, the WSL environment where vLLM, GuideLLM and lm_eval are installed.
- Stop any other vLLM server first, because this notebook starts its own server on port 8000.
- lm_eval downloads its benchmark datasets from the Hugging Face Hub on the first run, so you need an internet connection. Later runs use the local cache.

## Step 1: Imports and settings

- **`MODELS`**: the folders of the two models to compare.
- **`TOKENIZER`**: both tools count tokens with a tokenizer. The BF16 folder's tokenizer is used for both models, because quantization doesn't change the tokenizer and the GPTQ folder's `tokenizer_config.json` is saved in a newer Transformers format that this environment can't read.
- The environment variables turn off progress bars, so the saved notebook output stays short.

In [1]:
import os
os.environ["TQDM_DISABLE"] = "1"                    # no progress bars from lm_eval / datasets
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["OPENAI_API_KEY"] = "unused"             # lm_eval's API client expects a key; vLLM ignores it

import warnings, logging
warnings.filterwarnings("ignore")
for name in ["lm_eval", "datasets", "httpx", "urllib3"]:
    logging.getLogger(name).setLevel(logging.ERROR)

import atexit, json, re, signal, subprocess, sys, time
from pathlib import Path

import requests
from openai import OpenAI
from IPython.display import Markdown, display

MODELS_DIR = Path.cwd() / "Models"
MODELS = {
    "qwen3-0.6b-bf16":       MODELS_DIR / "Qwen3-0.6B",
    "qwen3-0.6b-gptq-w8a16": MODELS_DIR / "Qwen3-0.6B-GPTQ-W8A16",
}
BF16, GPTQ = list(MODELS)
TOKENIZER = MODELS[BF16]

PORT = 8000
VLLM_URL = f"http://localhost:{PORT}"
GPU_MEMORY_UTILIZATION = 0.75
MAX_MODEL_LEN = 4096

LOG_DIR = Path("logs");                   LOG_DIR.mkdir(exist_ok=True)
GUIDELLM_DIR = Path("outputs/guidellm");  GUIDELLM_DIR.mkdir(parents=True, exist_ok=True)
LM_EVAL_DIR = Path("outputs/lm_eval");    LM_EVAL_DIR.mkdir(parents=True, exist_ok=True)

perf = {}      # GuideLLM results:  perf[model][profile] = {...}
quality = {}   # lm_eval results:   quality[model][task]  = {...}

for name, path in MODELS.items():
    print(f"{name:<24} {path.relative_to(Path.cwd())}  ->  {'found' if path.exists() else 'NOT FOUND'}")

qwen3-0.6b-bf16          Models/Qwen3-0.6B  ->  found
qwen3-0.6b-gptq-w8a16    Models/Qwen3-0.6B-GPTQ-W8A16  ->  found


## Step 2: Helpers to start and stop the vLLM server

`start_server()` runs `vllm serve` in the background with the settings above, writes its output to a log file in `logs/`, and waits until the server answers on `/v1/models`. `stop_server()` shuts it down and frees the GPU memory, so the next model can load. It also runs automatically when the kernel shuts down, so a failed cell never leaves a server holding the GPU.

`warm_up()` sends a few short requests before benchmarking. The first requests after startup are slower, and they would distort the results.

Only one model is served at a time, which keeps memory use within the 6 GB of an RTX 3050 laptop GPU.

In [2]:
VLLM_BIN = Path(sys.executable).parent / "vllm"   # the vllm command from this kernel's environment
server = None

def server_is_up():
    try:
        return requests.get(f"{VLLM_URL}/v1/models", timeout=2).status_code == 200
    except requests.ConnectionError:
        return False

def start_server(model_path, served_name, tokenizer=None, timeout=600):
    """Start `vllm serve` in the background and wait until it accepts requests."""
    global server
    if server_is_up():
        raise RuntimeError(f"Something is already serving on port {PORT}. "
                           "Stop it first (stop_server() or `pkill -f 'vllm serve'`).")
    cmd = [str(VLLM_BIN), "serve", str(model_path),
           "--served-model-name", served_name,
           "--port", str(PORT),
           "--max-model-len", str(MAX_MODEL_LEN),
           "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION)]
    if tokenizer:
        cmd += ["--tokenizer", str(tokenizer)]
    print("$ vllm", " ".join(cmd[1:]).replace(str(Path.cwd()) + "/", ""))

    log_path = LOG_DIR / f"vllm_{served_name}.log"
    server = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                              start_new_session=True)
    server.log_path = log_path
    start = time.time()
    while time.time() - start < timeout:
        if server.poll() is not None:
            print(log_path.read_text()[-3000:])
            raise RuntimeError(f"vLLM server exited. Full log: {log_path}")
        if server_is_up():
            print(f"Server ready after {time.time() - start:.0f}s  (log: {log_path})")
            return
        time.sleep(3)
    raise TimeoutError(f"Server not ready after {timeout}s. Check {log_path}")

def stop_server():
    """Stop the server and free its GPU memory."""
    global server
    if server is None or server.poll() is not None:
        print("No server running.")
        return
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    try:
        server.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(server.pid), signal.SIGKILL)
        server.wait()
    while server_is_up():
        time.sleep(1)
    print("Server stopped.")
    server = None

atexit.register(stop_server)   # also stop the server if the kernel shuts down or restarts

def warm_up(model, n=5):
    """Send a few short requests, so one-time startup costs don't skew the first benchmark."""
    client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")
    for _ in range(n):
        client.chat.completions.create(model=model, max_tokens=32,
            messages=[{"role": "user", "content": "Say hello."}],
            extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    print(f"Warm-up done ({n} requests).")

## Step 3: Performance benchmarking with GuideLLM

GuideLLM sends requests to the server under a controlled load and records the timing of every request and every token.

| Metric | What it measures | Better |
|:--|:--|:--|
| **TTFT** (time to first token) | How long the user waits before the answer starts | Lower |
| **ITL** (inter-token latency) | Time between streamed tokens, i.e. how smoothly text appears | Lower |
| **Request latency** | Total time for one full answer | Lower |
| **Output tokens/s** | Tokens the server generates per second across all requests | Higher |
| **Requests/s** | Completed requests per second | Higher |

**Load profiles.** Each model is tested under three levels of load:

| Profile | What happens | Shows |
|:--|:--|:--|
| `synchronous` | One request at a time | The best-case latency for a single user |
| `concurrent@4` | 4 requests always in flight | A light multi-user load |
| `concurrent@16` | 16 requests always in flight | A heavier load, where continuous batching pays off |

**Test data.** `synthetic_text` creates random prompts of a fixed length: 256 prompt tokens and 128 output tokens per request. Fixed lengths make the two models directly comparable.

**Percentiles.** Averages hide slow outliers, so the results show **p50** (the median request) and **p95** (only 5% of requests are slower). p95 is what your slowest users experience.

In [3]:
PROMPT_TOKENS, OUTPUT_TOKENS = 256, 128
PROFILES = {                          # label: (profile option, number of requests)
    "synchronous":   ("kind=synchronous",            30),
    "concurrent@4":  ("kind=concurrent,streams=4",   64),
    "concurrent@16": ("kind=concurrent,streams=16", 128),
}

def run_guidellm(model, label):
    """Run one GuideLLM benchmark and return the path of its JSON report."""
    profile, n_requests = PROFILES[label]
    out = GUIDELLM_DIR / f"{model}_{label.replace('@', '_c')}.json"
    cmd = ["guidellm", "run",
           "--backend",    f"kind=openai_http,target={VLLM_URL},model={model}",
           "--profile",    profile,
           "--constraint", f"kind=max_requests,count={n_requests}",
           "--data",       f"kind=synthetic_text,prompt_tokens={PROMPT_TOKENS},output_tokens={OUTPUT_TOKENS}",
           "--tokenizer",  f"kind=hf_auto,model={TOKENIZER}",
           "--output",     f"kind=json,path={out}",
           "--disable-console"]
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    if result.returncode != 0:
        print(result.stdout[-2000:], result.stderr[-2000:])
        raise RuntimeError(f"GuideLLM failed for {model} / {label}")
    print(f"  {label:<14} {n_requests:>4} requests  done in {time.time() - start:5.1f}s  -> {out}")
    return out

GuideLLM's JSON report stores a full set of statistics (mean, percentiles, min, max) for every metric. `summarize_guidellm()` picks out the numbers used in this notebook:

In [4]:
def summarize_guidellm(path):
    bench = json.loads(Path(path).read_text())["benchmarks"][0]
    m = bench["metrics"]
    ok = lambda key: m[key]["successful"]
    return {
        "requests":        m["request_totals"]["successful"],
        "errors":          m["request_totals"]["errored"],
        "concurrency":     ok("request_concurrency")["mean"],
        "ttft_p50_ms":     ok("time_to_first_token_ms")["percentiles"]["p50"],
        "ttft_p95_ms":     ok("time_to_first_token_ms")["percentiles"]["p95"],
        "itl_p50_ms":      ok("inter_token_latency_ms")["percentiles"]["p50"],
        "itl_p95_ms":      ok("inter_token_latency_ms")["percentiles"]["p95"],
        "latency_p50_s":   ok("request_latency")["percentiles"]["p50"],
        "latency_p95_s":   ok("request_latency")["percentiles"]["p95"],
        "output_tok_s":    ok("output_tokens_per_second")["mean"],
        "requests_s":      ok("requests_per_second")["mean"],
    }

PERF_COLUMNS = [("Profile", None), ("Requests", "requests"), ("Errors", "errors"),
                ("TTFT p50 (ms)", "ttft_p50_ms"), ("TTFT p95 (ms)", "ttft_p95_ms"),
                ("ITL p50 (ms)", "itl_p50_ms"), ("ITL p95 (ms)", "itl_p95_ms"),
                ("Latency p50 (s)", "latency_p50_s"), ("Latency p95 (s)", "latency_p95_s"),
                ("Output tok/s", "output_tok_s"), ("Requests/s", "requests_s")]

def num(v):
    return f"{v:,}" if isinstance(v, int) else f"{v:,.3g}" if abs(v) < 10 else f"{v:,.1f}"

def show_perf(model):
    rows = ["| " + " | ".join(c for c, _ in PERF_COLUMNS) + " |",
            "|" + ":--|" + "--:|" * (len(PERF_COLUMNS) - 1)]
    for label, s in perf[model].items():
        rows.append(f"| **{label}** | " + " | ".join(num(s[k]) for _, k in PERF_COLUMNS[1:]) + " |")
    display(Markdown(f"**{model}**\n\n" + "\n".join(rows)))

## Step 4: Quality evaluation with lm_eval

lm_eval runs standard benchmark tasks and scores the answers. It uses the `local-completions` backend, which talks to vLLM's OpenAI-compatible `/v1/completions` endpoint.

**How multiple-choice scoring works.** For each question, lm_eval asks the server for the **log-probability** of every possible answer. The answer the model finds most likely counts as its choice. No text is generated, so scoring is fast and repeatable.

| Task | What it tests | Choices | Metric |
|:--|:--|:--:|:--|
| [HellaSwag](https://rowanzellers.com/hellaswag/) | Picking the most sensible ending to an everyday scenario | 4 | `acc_norm` |
| [ARC-Easy](https://allenai.org/data/arc) | Grade-school science questions | 4 | `acc` |
| [ARC-Challenge](https://allenai.org/data/arc) | Harder science questions that simple word matching can't solve | 4 | `acc_norm` |
| [WinoGrande](https://winogrande.allenai.org/) | Working out what a pronoun refers to, using common sense | 2 | `acc` |

- **`acc`** counts how often the most likely answer is correct.
- **`acc_norm`** divides each answer's log-probability by its length first, so longer answers aren't penalized. It's the standard metric for HellaSwag and ARC-Challenge.
- **`LIMIT`** evaluates the first 1,000 questions of each task. That's enough for a stable comparison, with a standard error of about ±1.5 percentage points. Set it to `None` to run the full test sets.
- The tasks run **0-shot**, meaning no worked examples are added to the prompt. Published leaderboards often use few-shot prompts, so their numbers are not directly comparable to these.

In [5]:
import contextlib, io
import lm_eval
from concurrent.futures import ThreadPoolExecutor

TASKS = {"hellaswag": "acc_norm", "arc_easy": "acc", "arc_challenge": "acc_norm", "winogrande": "acc"}
LIMIT = 1000
NUM_CONCURRENT = 16        # parallel requests to the server

def run_lm_eval(model):
    """Evaluate the served model on all TASKS and return per-task results."""
    start = time.time()
    # lm_eval sends parallel requests with asyncio. Jupyter already runs its own asyncio event loop,
    # so lm_eval runs in a separate thread, which has no event loop of its own.
    # Its "Tokenizing inputs" progress bar goes to stderr, which is hidden here to keep the output short.
    with ThreadPoolExecutor(max_workers=1) as pool, contextlib.redirect_stderr(io.StringIO()):
        out = pool.submit(lm_eval.simple_evaluate, **eval_args(model)).result()
    print(f"lm_eval finished in {time.time() - start:.0f}s\n")
    return parse_lm_eval(model, out)

def eval_args(model):
    return dict(
        model="local-completions",
        model_args=(f"model={model},"
                    f"base_url={VLLM_URL}/v1/completions,"
                    f"tokenizer={TOKENIZER},"
                    "tokenized_requests=False,"
                    f"num_concurrent={NUM_CONCURRENT}"),
        tasks=list(TASKS),
        limit=LIMIT,
    )

def parse_lm_eval(model, out):
    """Pull the score, error and per-question picks for each task, and save the raw results."""
    results = {}
    for task, metric in TASKS.items():
        r = out["results"][task]
        # the answer the model picked for each question: the choice with the highest log-probability
        picks = {s["doc_id"]: max(range(len(s["filtered_resps"])), key=lambda i: s["filtered_resps"][i][0])
                 for s in out["samples"][task]}
        results[task] = {"metric": metric,
                         "score": r[f"{metric},none"],
                         "stderr": r[f"{metric}_stderr,none"],
                         "n": out["n-samples"][task]["effective"],
                         "picks": picks}

    (LM_EVAL_DIR / f"{model}.json").write_text(json.dumps(
        {"results": out["results"], "n-samples": out["n-samples"], "lm_eval_version": out.get("lm_eval_version")},
        indent=2, default=str))
    return results

def show_quality(model):
    rows = ["| Task | Metric | Questions | Score |", "|:--|:--|--:|--:|"]
    for task, r in quality[model].items():
        rows.append(f"| {task} | `{r['metric']}` | {r['n']:,} | {r['score'] * 100:.1f}% ± {r['stderr'] * 100:.1f} |")
    display(Markdown(f"**{model}**\n\n" + "\n".join(rows)))

## Step 5: Benchmark and evaluate the BF16 model

### Start the server

In [6]:
start_server(MODELS[BF16], BF16)
warm_up(BF16)

$ vllm serve Models/Qwen3-0.6B --served-model-name qwen3-0.6b-bf16 --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.75


Server ready after 51s  (log: logs/vllm_qwen3-0.6b-bf16.log)


Warm-up done (5 requests).


### Run GuideLLM

The three load profiles run one after the other. Each one saves a JSON report in `outputs/guidellm/`.

In [7]:
perf[BF16] = {}
for label in PROFILES:
    perf[BF16][label] = summarize_guidellm(run_guidellm(BF16, label))
show_perf(BF16)

  synchronous      30 requests  done in  47.5s  -> outputs/guidellm/qwen3-0.6b-bf16_synchronous.json


  concurrent@4     64 requests  done in  34.8s  -> outputs/guidellm/qwen3-0.6b-bf16_concurrent_c4.json


  concurrent@16   128 requests  done in  26.8s  -> outputs/guidellm/qwen3-0.6b-bf16_concurrent_c16.json


**qwen3-0.6b-bf16**

| Profile | Requests | Errors | TTFT p50 (ms) | TTFT p95 (ms) | ITL p50 (ms) | ITL p95 (ms) | Latency p50 (s) | Latency p95 (s) | Output tok/s | Requests/s |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| **synchronous** | 30 | 0 | 44.5 | 52.1 | 9.25 | 9.28 | 1.22 | 1.23 | 104.0 | 0.813 |
| **concurrent@4** | 64 | 0 | 62.1 | 254.0 | 11.4 | 11.8 | 1.54 | 1.71 | 330.8 | 2.58 |
| **concurrent@16** | 128 | 0 | 136.1 | 346.5 | 15.2 | 16.3 | 2.17 | 2.21 | 975.8 | 7.62 |

**How to read this table:**
- From `synchronous` to `concurrent@16`, **output tokens/s** should grow a lot. That's continuous batching: the GPU generates one token for many requests in the same step.
- **ITL** and **latency** grow with load too, because each step now serves more requests. That's the trade-off between throughput and per-user speed.
- **TTFT** rises under load because new requests wait for the prompt processing of others.

### Run lm_eval

This sends about 14,000 log-probability requests, so it's the longest step. The first run also downloads the datasets.

In [8]:
quality[BF16] = run_lm_eval(BF16)
show_quality(BF16)

lm_eval finished in 220s



**qwen3-0.6b-bf16**

| Task | Metric | Questions | Score |
|:--|:--|--:|--:|
| hellaswag | `acc_norm` | 1,000 | 47.8% ± 1.6 |
| arc_easy | `acc` | 1,000 | 60.8% ± 1.5 |
| arc_challenge | `acc_norm` | 1,000 | 34.0% ± 1.5 |
| winogrande | `acc` | 1,000 | 56.7% ± 1.6 |

Random guessing would score 25% on the 4-choice tasks and 50% on WinoGrande. A 0.6B model sits clearly above chance, but far below large models.

Stop the server to free GPU memory for the next model:

In [9]:
stop_server()

Server stopped.


## Step 6: Benchmark and evaluate the GPTQ-W8A16 model

The same steps run on the quantized model. **W8A16** means the weights of the linear layers are stored as 8-bit integers (half the size of BF16), while activations stay 16-bit. vLLM reads the format from `config.json` and picks a fast 8-bit kernel automatically.

`tokenizer=` points to the BF16 folder, for the reason explained in Step 1.

### Start the server

In [10]:
start_server(MODELS[GPTQ], GPTQ, tokenizer=TOKENIZER)
warm_up(GPTQ)

$ vllm serve Models/Qwen3-0.6B-GPTQ-W8A16 --served-model-name qwen3-0.6b-gptq-w8a16 --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.75 --tokenizer Models/Qwen3-0.6B


Server ready after 48s  (log: logs/vllm_qwen3-0.6b-gptq-w8a16.log)


Warm-up done (5 requests).


### Run GuideLLM

In [11]:
perf[GPTQ] = {}
for label in PROFILES:
    perf[GPTQ][label] = summarize_guidellm(run_guidellm(GPTQ, label))
show_perf(GPTQ)

  synchronous      30 requests  done in  37.3s  -> outputs/guidellm/qwen3-0.6b-gptq-w8a16_synchronous.json


  concurrent@4     64 requests  done in  29.7s  -> outputs/guidellm/qwen3-0.6b-gptq-w8a16_concurrent_c4.json


  concurrent@16   128 requests  done in  24.9s  -> outputs/guidellm/qwen3-0.6b-gptq-w8a16_concurrent_c16.json


**qwen3-0.6b-gptq-w8a16**

| Profile | Requests | Errors | TTFT p50 (ms) | TTFT p95 (ms) | ITL p50 (ms) | ITL p95 (ms) | Latency p50 (s) | Latency p95 (s) | Output tok/s | Requests/s |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| **synchronous** | 30 | 0 | 44.2 | 53.1 | 6.76 | 6.8 | 0.903 | 0.912 | 139.6 | 1.09 |
| **concurrent@4** | 64 | 0 | 67.0 | 292.2 | 8.99 | 9.32 | 1.24 | 1.41 | 413.5 | 3.23 |
| **concurrent@16** | 128 | 0 | 141.6 | 266.9 | 13.0 | 14.0 | 1.8 | 1.94 | 1,143.5 | 8.86 |

### Run lm_eval

In [12]:
quality[GPTQ] = run_lm_eval(GPTQ)
show_quality(GPTQ)

lm_eval finished in 206s



**qwen3-0.6b-gptq-w8a16**

| Task | Metric | Questions | Score |
|:--|:--|--:|--:|
| hellaswag | `acc_norm` | 1,000 | 48.1% ± 1.6 |
| arc_easy | `acc` | 1,000 | 60.9% ± 1.5 |
| arc_challenge | `acc_norm` | 1,000 | 34.3% ± 1.5 |
| winogrande | `acc` | 1,000 | 55.7% ± 1.6 |

In [13]:
stop_server()

Server stopped.


## Step 7: Compare performance

For each load profile, the table shows the BF16 value, the GPTQ-W8A16 value and the change. For latency a negative change is better, and for throughput a positive change is better.

In [14]:
COMPARE_PERF = [("TTFT p50 (ms)", "ttft_p50_ms", "lower"), ("TTFT p95 (ms)", "ttft_p95_ms", "lower"),
                ("ITL p50 (ms)", "itl_p50_ms", "lower"), ("ITL p95 (ms)", "itl_p95_ms", "lower"),
                ("Latency p50 (s)", "latency_p50_s", "lower"), ("Latency p95 (s)", "latency_p95_s", "lower"),
                ("Output tok/s", "output_tok_s", "higher"), ("Requests/s", "requests_s", "higher")]

def change(a, b, better):
    """Percent change from BF16 (a) to GPTQ (b). `better` is kept so callers state which direction is good."""
    return f"{(b - a) / a * 100:+.0f}%"

for label in PROFILES:
    a, b = perf[BF16][label], perf[GPTQ][label]
    rows = ["| Metric | BF16 | GPTQ-W8A16 | Change |", "|:--|--:|--:|--:|"]
    rows += [f"| {name} | {num(a[k])} | {num(b[k])} | {change(a[k], b[k], better)} |"
             for name, k, better in COMPARE_PERF]
    display(Markdown(f"#### {label}\n\n" + "\n".join(rows)))

#### synchronous

| Metric | BF16 | GPTQ-W8A16 | Change |
|:--|--:|--:|--:|
| TTFT p50 (ms) | 44.5 | 44.2 | -1% |
| TTFT p95 (ms) | 52.1 | 53.1 | +2% |
| ITL p50 (ms) | 9.25 | 6.76 | -27% |
| ITL p95 (ms) | 9.28 | 6.8 | -27% |
| Latency p50 (s) | 1.22 | 0.903 | -26% |
| Latency p95 (s) | 1.23 | 0.912 | -26% |
| Output tok/s | 104.0 | 139.6 | +34% |
| Requests/s | 0.813 | 1.09 | +34% |

#### concurrent@4

| Metric | BF16 | GPTQ-W8A16 | Change |
|:--|--:|--:|--:|
| TTFT p50 (ms) | 62.1 | 67.0 | +8% |
| TTFT p95 (ms) | 254.0 | 292.2 | +15% |
| ITL p50 (ms) | 11.4 | 8.99 | -21% |
| ITL p95 (ms) | 11.8 | 9.32 | -21% |
| Latency p50 (s) | 1.54 | 1.24 | -20% |
| Latency p95 (s) | 1.71 | 1.41 | -17% |
| Output tok/s | 330.8 | 413.5 | +25% |
| Requests/s | 2.58 | 3.23 | +25% |

#### concurrent@16

| Metric | BF16 | GPTQ-W8A16 | Change |
|:--|--:|--:|--:|
| TTFT p50 (ms) | 136.1 | 141.6 | +4% |
| TTFT p95 (ms) | 346.5 | 266.9 | -23% |
| ITL p50 (ms) | 15.2 | 13.0 | -14% |
| ITL p95 (ms) | 16.3 | 14.0 | -14% |
| Latency p50 (s) | 2.17 | 1.8 | -17% |
| Latency p95 (s) | 2.21 | 1.94 | -12% |
| Output tok/s | 975.8 | 1,143.5 | +17% |
| Requests/s | 7.62 | 8.86 | +16% |

**Why the 8-bit model is faster.** Generating each token is **memory-bound**: the GPU spends most of its time reading the model's weights from memory, not doing math. 8-bit weights are half the size, so each step reads about half as much data. The gain shows up most in **ITL** and single-user speed.

As load grows, more of each step goes to attention and batching work that quantization doesn't speed up, so the relative gain usually shrinks.

**Why TTFT barely changes.** TTFT is mostly **prompt processing** (prefill): all 256 prompt tokens go through the model in one large matrix multiplication. That work is **compute-bound**, and W8A16 converts the 8-bit weights back to 16-bit to do the math, so prefill is no faster than BF16. Under load, TTFT also depends on when a request arrives relative to other requests' prefill, so with 64–128 requests a TTFT difference of a few percent, or even 10–20% at p95, is within normal run-to-run noise. Formats that also quantize activations, such as W8A8, can speed up prefill as well.

## Step 8: Compare quality

Two views of quality:

- **Recovery** = quantized score ÷ BF16 score. 100% means no accuracy was lost. Above about 99% is generally considered lossless.
- **Agreement**: the share of questions where both models picked the **same** answer, whether right or wrong. It's a stricter test than accuracy, because two models can reach the same score by getting *different* questions right.

Score differences smaller than the ± standard error are within normal measurement noise.

In [15]:
rows = ["| Task | Metric | BF16 | GPTQ-W8A16 | Difference | Recovery | Agreement |",
        "|:--|:--|--:|--:|--:|--:|--:|"]
recoveries, agreements = [], []
for task in TASKS:
    a, b = quality[BF16][task], quality[GPTQ][task]
    recovery = b["score"] / a["score"] * 100
    shared = a["picks"].keys() & b["picks"].keys()
    agree = sum(a["picks"][d] == b["picks"][d] for d in shared) / len(shared) * 100
    recoveries.append(recovery); agreements.append(agree)
    rows.append(f"| {task} | `{a['metric']}` | {a['score'] * 100:.1f}% ± {a['stderr'] * 100:.1f} "
                f"| {b['score'] * 100:.1f}% ± {b['stderr'] * 100:.1f} | {(b['score'] - a['score']) * 100:+.1f} pts "
                f"| {recovery:.1f}% | {agree:.1f}% |")

avg_a = sum(quality[BF16][t]["score"] for t in TASKS) / len(TASKS) * 100
avg_b = sum(quality[GPTQ][t]["score"] for t in TASKS) / len(TASKS) * 100
rows.append(f"| **Average** | | **{avg_a:.1f}%** | **{avg_b:.1f}%** | **{avg_b - avg_a:+.1f} pts** "
            f"| **{avg_b / avg_a * 100:.1f}%** | **{sum(agreements) / len(agreements):.1f}%** |")
display(Markdown("\n".join(rows)))

| Task | Metric | BF16 | GPTQ-W8A16 | Difference | Recovery | Agreement |
|:--|:--|--:|--:|--:|--:|--:|
| hellaswag | `acc_norm` | 47.8% ± 1.6 | 48.1% ± 1.6 | +0.3 pts | 100.6% | 99.2% |
| arc_easy | `acc` | 60.8% ± 1.5 | 60.9% ± 1.5 | +0.1 pts | 100.2% | 97.0% |
| arc_challenge | `acc_norm` | 34.0% ± 1.5 | 34.3% ± 1.5 | +0.3 pts | 100.9% | 98.3% |
| winogrande | `acc` | 56.7% ± 1.6 | 55.7% ± 1.6 | -1.0 pts | 98.2% | 96.4% |
| **Average** | | **49.8%** | **49.8%** | **-0.1 pts** | **99.8%** | **97.7%** |

## Step 9: Summary of the trade-off

This cell puts the key numbers together: how much smaller and faster the quantized model is, and how much accuracy it keeps. All results are also saved to `outputs/benchmark_eval_results.json`.

In [16]:
size = lambda m: sum(f.stat().st_size for f in MODELS[m].glob("*.safetensors")) / 1e9
sync_a, sync_b = perf[BF16]["synchronous"], perf[GPTQ]["synchronous"]
load_a, load_b = perf[BF16]["concurrent@16"], perf[GPTQ]["concurrent@16"]

summary = {
    "size_gb":              (size(BF16), size(GPTQ)),
    "single_user_tok_s":    (sync_a["output_tok_s"], sync_b["output_tok_s"]),
    "itl_p50_ms":           (sync_a["itl_p50_ms"], sync_b["itl_p50_ms"]),
    "throughput_16_tok_s":  (load_a["output_tok_s"], load_b["output_tok_s"]),
    "avg_accuracy_pct":     (avg_a, avg_b),
}
rows = ["| | BF16 | GPTQ-W8A16 | Change |", "|:--|--:|--:|--:|",
        f"| Model size on disk (GB) | {summary['size_gb'][0]:.2f} | {summary['size_gb'][1]:.2f} | {change(*summary['size_gb'], 'lower')} |",
        f"| Single-user speed (tok/s) | {num(sync_a['output_tok_s'])} | {num(sync_b['output_tok_s'])} | {change(*summary['single_user_tok_s'], 'higher')} |",
        f"| Inter-token latency p50 (ms) | {num(sync_a['itl_p50_ms'])} | {num(sync_b['itl_p50_ms'])} | {change(*summary['itl_p50_ms'], 'lower')} |",
        f"| Throughput at 16 concurrent (tok/s) | {num(load_a['output_tok_s'])} | {num(load_b['output_tok_s'])} | {change(*summary['throughput_16_tok_s'], 'higher')} |",
        f"| Average accuracy (4 tasks) | {avg_a:.1f}% | {avg_b:.1f}% | {avg_b - avg_a:+.1f} pts |",
        f"| Answer agreement with BF16 | 100% | {sum(agreements) / len(agreements):.1f}% | |"]
display(Markdown("\n".join(rows)))

results_path = Path("outputs/benchmark_eval_results.json")
results_path.write_text(json.dumps({
    "settings": {"prompt_tokens": PROMPT_TOKENS, "output_tokens": OUTPUT_TOKENS,
                 "profiles": {k: {"profile": v[0], "requests": v[1]} for k, v in PROFILES.items()},
                 "tasks": TASKS, "limit": LIMIT},
    "performance": perf,
    "quality": {m: {t: {k: v for k, v in r.items() if k != "picks"} for t, r in q.items()} for m, q in quality.items()},
    "summary": summary,
}, indent=2))
print(f"Saved to {results_path}")

| | BF16 | GPTQ-W8A16 | Change |
|:--|--:|--:|--:|
| Model size on disk (GB) | 1.50 | 0.76 | -50% |
| Single-user speed (tok/s) | 104.0 | 139.6 | +34% |
| Inter-token latency p50 (ms) | 9.25 | 6.76 | -27% |
| Throughput at 16 concurrent (tok/s) | 975.8 | 1,143.5 | +17% |
| Average accuracy (4 tasks) | 49.8% | 49.8% | -0.1 pts |
| Answer agreement with BF16 | 100% | 97.7% | |

Saved to outputs/benchmark_eval_results.json


**How to decide.** A quantized model is a good choice when it is clearly faster or smaller *and* its accuracy stays within measurement noise of the original:

- If **recovery is about 99% or higher** and **agreement is high**, the quantized model behaves like the original and the speed and memory savings come almost for free.
- If accuracy drops on the tasks that matter for your application, try a less aggressive format, or check whether the drop is larger than the ± error before deciding.
- The memory saved also matters on a small GPU: smaller weights leave more room for KV cache, which means more requests or longer prompts at the same time.


**Resources:** [GuideLLM](https://github.com/vllm-project/guidellm) · [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) · [vLLM quantization docs](https://docs.vllm.ai/en/latest/features/quantization/)